In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
from poke_env.player import RandomPlayer, SimpleHeuristicsPlayer, MaxBasePowerPlayer
from poke_env.player.battle_order import BattleOrder, SingleBattleOrder
from agent.state import State
from agent.player import BotBoi
from agent.model import DQN
from agent.train import DQNTrainer
from agent.memory import ExperienceReplay

import os
import psycopg2
from dotenv import load_dotenv

In [4]:
# from db.connection import clear_database
# load_dotenv()

# conn = psycopg2.connect(
#     host=os.getenv("DB_HOST"),
#     port=int(os.getenv("DB_PORT")),
#     dbname=os.getenv("DB_NAME"),
#     user=os.getenv("DB_USER"),
#     password=os.getenv("DB_PASSWORD")
#     )

# clear_database(conn)

# conn.close()

In [5]:
load_dotenv()

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD")
    )

model = DQN(hidden_sizes=[512, 256, 256, 64]).to(device)
model.load_state_dict(torch.load("./checkpoints/dqn_v1.pth"))
agent = BotBoi(model, device)
optim = torch.optim.Adam(model.parameters(), lr=1e-4)
opponent = SimpleHeuristicsPlayer(battle_format="gen9randombattle")
memory = ExperienceReplay()

trainer = DQNTrainer(agent, opponent, memory, optim, device, conn=conn, model_checkpoint="./checkpoints/dqn_v1.pth", epsilon=0.20, gamma=0.90, lr=1e-4)
await trainer.train_model(num_episodes=20000)

conn.close()

In [ ]:
# conn.close()